## Wards

## Wards (December 2025, UK BFC) -- England, Scotland, Wales only

**Note on scope:** wards are not a single clean "electoral unit" the way constituencies are, across all four nations:
- **England / Wales** : wards are the direct electoral unit for local councils (1-3 councillors each)
- **Scotland** : wards are multi-member (3-4 councillors, STV); same concept, different structure
- **Northern Ireland** : wards exist but are *not* the electoral unit local elections use larger **District Electoral Areas (DEAs)**, which group several wards together. NI wards are mainly a statistical/admin building block underneath DEAs, not something voted in directly. 

Source: `WD_DEC_2025_UK_BFC` -- https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/WD_DEC_2025_UK_BFC/FeatureServer/0

In [13]:
import requests
import geopandas as gpd
import pandas as pd
import json
import time

def fetch_arcgis_layer(base_url, page_size=200, timeout=120, max_retries=3):
    offset = 0
    frames = []
    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": page_size,
        }
        for attempt in range(max_retries):
            try:
                r = requests.get(f"{base_url}/query", params=params, timeout=timeout)
                r.raise_for_status()
                data = r.json()
                break
            except (requests.exceptions.RequestException, requests.exceptions.ChunkedEncodingError) as e:
                if attempt == max_retries - 1:
                    raise
                wait = 5 * (attempt + 1)
                print(f"Page at offset {offset}, attempt {attempt+1} failed ({type(e).__name__}), retrying in {wait}s...")
                time.sleep(wait)

        if not data.get("features"):
            break
        frames.append(gpd.GeoDataFrame.from_features(data["features"]))
        print(f"Fetched {offset + len(data['features'])} features so far...")
        if len(data["features"]) < page_size:
            break
        offset += page_size
    return pd.concat(frames, ignore_index=True)




**Note -- large-dataset fetch failure:** the standard
`fetch_arcgis_layer` (page_size=2000) failed partway through with
`ChunkedEncodingError: IncompleteRead(16777216 bytes read, 33916719 more
expected)`. This is a volume problem, not a permissions/endpoint problem
-- wards is a much bigger dataset than any constituency layer (~8,400
features vs. hundreds), so a 2000-record page at full BFC resolution
produces a payload large enough (tens of MB) that the connection dropped
mid-stream.

**Fix:** smaller pages (200 instead of 2000), longer timeout, and
retry-with-backoff so one dropped connection doesn't kill the whole pull.

In [14]:

wards_dec25 = fetch_arcgis_layer(
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/"
    "WD_DEC_2025_UK_BFC/FeatureServer/0",
    page_size=200,
)

Fetched 200 features so far...
Fetched 400 features so far...
Fetched 600 features so far...
Fetched 800 features so far...
Fetched 1000 features so far...
Fetched 1200 features so far...
Fetched 1400 features so far...
Fetched 1600 features so far...
Fetched 1800 features so far...
Fetched 2000 features so far...
Fetched 2200 features so far...
Fetched 2400 features so far...
Fetched 2600 features so far...
Fetched 2800 features so far...
Fetched 3000 features so far...
Fetched 3200 features so far...
Fetched 3400 features so far...
Fetched 3600 features so far...
Fetched 3800 features so far...
Fetched 4000 features so far...
Fetched 4200 features so far...
Fetched 4400 features so far...
Fetched 4600 features so far...
Fetched 4800 features so far...
Fetched 5000 features so far...
Fetched 5200 features so far...
Fetched 5400 features so far...
Fetched 5600 features so far...
Fetched 5800 features so far...
Fetched 6000 features so far...
Fetched 6200 features so far...
Fetched 6400

In [17]:
print(wards_dec25.columns)

Index(['geometry', 'FID', 'WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM',
       'BNG_E', 'BNG_N', 'LONG', 'LAT', 'Shape__Area', 'Shape__Length',
       'GlobalID'],
      dtype='str')


**Note:** original columns are
`['geometry', 'FID', 'WD25CD', 'WD25NM', 'WD25NMW', 'LAD25CD', 'LAD25NM', 'BNG_E', 'BNG_N', 'LONG', 'LAT', 'Shape__Area', 'Shape__Length', 'GlobalID']`

Two fields not seen in the constituency layers:
- `WD25NMW` -- Welsh name, same idea as Westminster's `PCON24NMW`
- `LAD25CD`/`LAD25NM` -- the parent Local Authority District each ward
  sits in. Useful (wards are always described relative to their council)
  but is a join key to a *different* geography table - allows connection with LAD dataset.

In [18]:
wards = wards_dec25.drop(columns=["FID", "Shape__Area", "Shape__Length"])

column_rename = {
    "WD25CD":   "ward_code",
    "WD25NM":   "ward_name",
    "WD25NMW":  "ward_name_welsh",
    "LAD25CD":  "local_authority_code",
    "LAD25NM":  "local_authority_name",
    "BNG_E":    "easting_bng",
    "BNG_N":    "northing_bng",
    "LONG":     "longitude",
    "LAT":      "latitude",
    "GlobalID": "global_id",
}

wards = wards.rename(columns=column_rename)

In [9]:
wards_metadata = {
    "layer": "wards_dec2025",
    "source_name": "WD_DEC_2025_UK_BFC \u2014 Wards (December 2025)",
    "source_url": "https://services1.arcgis.com/ESMARspQHYMw9BZ9/ArcGIS/rest/services/WD_DEC_2025_UK_BFC/FeatureServer/0",
    "publisher": "Office for National Statistics (ONS) Open Geography Portal",
    "boundary_type": "BFC \u2014 Full resolution, clipped to coastline (Mean High Water mark)",
    "as_at_date": "2025-12-01",
    "retrieved_date": "2026-07-20",
    "review_context": "Electoral wards for local government councils in England, Scotland, Northern Ireland and Wales. NI wards are a statistical unit only, not the electoral unit (District Electoral Areas are), per project scoping decision.",
    "crs": str(wards.crs),
    "n_features": len(wards),
    "dropped_fields": ["FID", "Shape__Area", "Shape__Length"],
    "original_to_renamed_columns": column_rename,
    "notes": "Includes LAD25CD/LAD25NM parent Local Authority District as a join key -- check for redundancy if an LAD layer already exists elsewhere in the catalogue. WD25 field prefix confirms December 2025 vintage; a separate WD26CD-prefixed lookup table exists on the same host for a later release -- do not mix vintages when joining.",
}

In [30]:
wards = gpd.GeoDataFrame(wards).set_crs(epsg=4326).to_crs(epsg=27700).copy()

In [31]:
wards[['ward_code', 'ward_name', 'ward_name_welsh', 'local_authority_code', 'local_authority_name', 'easting_bng', 'northing_bng','longitude','latitude','global_id','geometry']].to_file(r"C:\Users\spspa\OneDrive - The University of Liverpool\geographies\data\processed\wards_dec2025.gpkg", layer="wards_dec2025", driver="GPKG")